# 17. Routing Module — classification-based dispatch

When one adapter isn't enough. `RoutingModule` lets a single `invoke()`
call land on a different provider+model based on a `classification`
kwarg passed alongside the messages. Sensitive workloads route to one
FedRAMP-cleared backend; cheap chit-chat routes to a fast, low-cost one.
Same caller, same prompt shape — different destination.

This notebook is the deep dive on `RoutingModule`. Notebook **04**
showed it as one of sixteen kwargs to `load_model`; notebook **06**
showed how the registry wires it in. Here we look inside.

**You will learn:**

- Why classification-based dispatch matters for federal and multi-tenant deployments
- The exact config shape `RoutingModule` accepts and what gets validated at construction
- How the `classification` kwarg flows through the module stack and gets popped before reaching the adapter
- Why classification strings are constrained to a strict ASCII regex (homoglyph defense)
- The two enforcement modes — `"block"` and `"warn"` — and exactly what each does on a miss
- How to compose a router under `TelemetryModule` and other stack layers
- How `close()` propagates to every adapter in the routing table

## 1. Setup

Standard Arc walkthrough boilerplate — locate the repo root and put every `packages/<pkg>/src` and `packages/<pkg>/tests` directory on `sys.path`. After this cell runs, plain imports like `from arcllm.modules.routing import RoutingModule` work without installing anything.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Sanity-check the imports we'll lean on throughout the notebook. `RoutingModule` is the only public name from `arcllm.modules.routing`; everything else (the regex, the logger) is private.

In [ ]:
from unittest.mock import AsyncMock, MagicMock

from arcllm.exceptions import ArcLLMConfigError
from arcllm.modules.routing import RoutingModule
from arcllm.types import LLMProvider, LLMResponse, Message, Tool, Usage

print("RoutingModule:", RoutingModule)
print("module:", RoutingModule.__module__)
print("base class:", RoutingModule.__bases__[0].__name__)

## 2. Why routing — multi-provider deployments

A single agent often needs more than one backend:

- **Cost vs quality.** Send routine summaries to Groq Llama; send
  the high-stakes analysis to Claude Opus. Same agent, same code path,
  10x price-per-token difference.
- **Data classification.** CUI (Controlled Unclassified Information)
  must land on a FedRAMP-authorized provider. Public chat can use
  any commercial endpoint. The router enforces the rule structurally
  — there is no opportunity for a developer to *forget* to switch
  models when handling sensitive data.
- **Tenant isolation.** Each tenant in a multi-tenant deployment can
  pin a specific model contract. Routing keys become tenant IDs.
- **Failover topology.** While `FallbackModule` handles failover on
  *errors*, `RoutingModule` handles intentional traffic shaping based
  on *intent*. The two compose.

The router lives at the **innermost** position in the module stack.
It replaces the single adapter with a switchboard. Every other
module — telemetry, retry, fallback, circuit breaker — wraps the
router as a unit, oblivious to which destination was chosen.

Picture the difference at a glance. Without routing the stack reads top-down to a single adapter; with routing, the bottom of the stack becomes a fan-out keyed on `classification`.

In [ ]:
topology = """
Without routing:                With routing:

    Telemetry                     Telemetry
        |                              |
      Retry                          Retry
        |                              |
    Fallback                       Fallback
        |                              |
   AnthropicAdapter             RoutingModule
                              /        |        \\
                          adapter["cui"]      adapter["unclassified"]
                          (Anthropic)       (OpenAI)
"""
print(topology)

## 3. `RoutingModule` basics — config shape

`RoutingModule.__init__` takes two arguments:

1. `config: dict[str, Any]` — runtime knobs:
   - `enforcement` — `"block"` (default) or `"warn"`. What to do
     when the request's classification doesn't match any rule.
   - `default_classification` — the string under which the
     fallback adapter is registered. Must exist in `adapters` or
     construction fails. Defaults to `"unclassified"`.
2. `adapters: dict[str, LLMProvider]` — a dict from classification
   string to a pre-built `LLMProvider` instance. Keys are the
   classification names you'll route on. Values are normal adapters
   (`AnthropicAdapter`, `OpenaiAdapter`, etc.) — already configured,
   already holding their own httpx pools.

Notice what this constructor *doesn't* do: it does not load TOML, it
does not resolve API keys, it does not build adapters. That work
happens in `load_model()` (see notebook 06). `RoutingModule` is
purely a switchboard over already-built adapters — clean separation
of concerns.

Here's the smallest possible mock adapter. We'll use this helper repeatedly, so define it once. It's a `MagicMock` typed against `LLMProvider` with `invoke` and `close` swapped for `AsyncMock` (so `await` works).

In [ ]:
_OK = LLMResponse(
    content="routed",
    usage=Usage(input_tokens=100, output_tokens=50, total_tokens=150),
    model="test-model",
    stop_reason="end_turn",
)


def make_adapter(name: str, model: str) -> MagicMock:
    """Build a mock adapter that returns _OK from invoke() and tracks calls."""
    adapter = MagicMock(spec=LLMProvider)
    adapter.name = name
    adapter.model_name = model
    adapter.validate_config.return_value = True
    adapter.invoke = AsyncMock(return_value=_OK)
    adapter.close = AsyncMock()
    return adapter

Build a router with two routes — `cui` to a Claude-shaped adapter, `unclassified` to a GPT-shaped adapter — and read back its identity properties.

In [ ]:
cui_adapter = make_adapter("anthropic", "claude-sonnet-4-6")
unc_adapter = make_adapter("openai", "gpt-4o-mini")

router = RoutingModule(
    {"enforcement": "block", "default_classification": "unclassified"},
    {"cui": cui_adapter, "unclassified": unc_adapter},
)

print("router.name        =", router.name)
print("router.model_name  =", router.model_name)
print("router.validate_config() =", router.validate_config())
print("\nName/model report the *default* adapter — the one keyed by")
print("default_classification. That's what callers see when they ask")
print("the router 'what are you?' before any invoke() decides routing.")

## 4. Classification model — where the tag comes from

The classification flows through the stack as a **kwarg** to
`invoke()`. The router pops it from `kwargs` and uses it to choose
an adapter; everything else passes through to the chosen adapter
untouched.

```python
await router.invoke(
    messages,
    tools,
    classification="cui",   # <-- consumed by RoutingModule
    max_tokens=500,         # <-- forwarded to the chosen adapter
)
```

Calling code is the source of truth. Whether that's:

- A static label per agent (set once at construction)
- A per-message classifier the agent runs upstream
- A field on a tenant context object propagated through the loop

...the router doesn't care. It only sees the kwarg. If you don't
pass one, the router uses `default_classification`.

Demonstrate kwarg-driven dispatch. Same router, same messages, three calls — three different routes.

In [ ]:
async def demo_kwarg_dispatch() -> None:
    msgs = [Message(role="user", content="hello")]

    # 1. Explicit classification → matching adapter
    await router.invoke(msgs, classification="cui")

    # 2. Explicit classification → matching adapter (the other route)
    await router.invoke(msgs, classification="unclassified")

    # 3. No classification → falls through to default_classification ("unclassified")
    await router.invoke(msgs)

    print("cui adapter was called", cui_adapter.invoke.await_count, "time(s)")
    print("unc adapter was called", unc_adapter.invoke.await_count, "time(s)")


# Reset call counts between demos so the report is clean
cui_adapter.invoke.reset_mock()
unc_adapter.invoke.reset_mock()
await demo_kwarg_dispatch()

Critical property: the `classification` kwarg is **popped**, not forwarded. Adapters never see it. They get the rest of the kwargs verbatim. Verify.

In [ ]:
async def demo_classification_popped() -> None:
    msgs = [Message(role="user", content="hi")]
    await router.invoke(msgs, classification="cui", max_tokens=42, temperature=0.1)

    # Inspect what the cui adapter actually received
    call_kwargs = cui_adapter.invoke.call_args.kwargs
    print("kwargs the adapter saw:", call_kwargs)
    assert "classification" not in call_kwargs, "classification should be popped!"
    assert call_kwargs["max_tokens"] == 42
    assert call_kwargs["temperature"] == 0.1
    print("\n[ok] classification stripped, other kwargs preserved")


cui_adapter.invoke.reset_mock()
await demo_classification_popped()

## 5. Classification format validation — strict ASCII

Classification strings are validated against a strict regex **before**
any adapter lookup happens:

```python
_CLASSIFICATION_RE = re.compile(r"^[a-z][a-z0-9_:.\-]{0,127}$")
```

Translation: must start with a lowercase ASCII letter, then 0–127
more characters from the set `[a-z0-9_:.-]`. **No uppercase. No
non-ASCII. No whitespace. Hard cap at 128 chars total.**

This is the **homoglyph defense**. Without it, a request labelled
with a *visually identical* but byte-different string — e.g.
fullwidth Latin `ｃｕｉ` (U+FF43 U+FF55 U+FF49) or Cyrillic `сui`
(Cyrillic `с` U+0441) — could slip past a naive `if classification
== "cui"` check. By rejecting any character outside ASCII lowercase
alphanumerics, the router refuses to even attempt the lookup. Hence
the term *NFKC-style* validation: equivalent to normalizing first
and then accepting only the canonical ASCII form, but implemented as
a deny-by-default regex (which is stricter and never silently
rewrites the input).

Validation runs **before** the adapter dict lookup. There is no path
by which a malformed string can reach an adapter.

First, the happy path — well-formed lowercase ASCII tags pass straight through.

In [ ]:
import re

# Same regex the module uses internally
RE = re.compile(r"^[a-z][a-z0-9_:.\-]{0,127}$")

valid_examples = [
    "cui",
    "unclassified",
    "tier1",
    "fed.high",
    "tenant:acme",
    "premium-fast",
    "long_classification_with_many_underscores",
    "a" + "b" * 127,  # exactly 128 chars — at the limit
]

for s in valid_examples:
    assert RE.match(s), f"expected to pass: {s!r}"
    print(f"  PASS  {s!r}")

Now the rejected forms. Each of these would be a homoglyph or injection attempt against a hand-rolled string-comparison check. The regex catches them all.

In [ ]:
# Build adversarial classification strings using \u escapes so the
# notebook source itself contains no ambiguous codepoints — only the
# runtime values do (which is the whole point of this defense).
fullwidth_premium = "\uff50\uff52\uff45\uff4d\uff49\uff55\uff4d"  # fullwidth p+r+e+m+i+u+m
cyrillic_cui = "\u0441ui"  # U+0441 (cyrillic ES) + ASCII u + ASCII i

rejected_examples = {
    "CUI": "uppercase letter",
    "Cui": "leading uppercase",
    "": "empty string",
    "1cui": "starts with digit",
    "_cui": "starts with underscore",
    "cui ": "trailing whitespace",
    "cui\n": "embedded newline",
    fullwidth_premium: "fullwidth Latin (NFKC homoglyph)",
    cyrillic_cui: "Cyrillic c (U+0441) homoglyph for ASCII c",
    "cui\u200b": "zero-width space injection",
    "cui;rm -rf /": "shell metacharacters",
    "a" + "b" * 128: "129 chars — over the length cap",
    "cui/path": "forward slash",
    "cui@tag": "at-sign",
}

for s, reason in rejected_examples.items():
    matched = bool(RE.match(s))
    status = "FAIL (regex passed!)" if matched else "REJECT"
    label = repr(s) if len(s) < 40 else f"{s[:30]!r}... ({len(s)} chars)"
    print(f"  {status:6}  {label:55}  ({reason})")

Now confirm `RoutingModule` actually enforces this — not just the
loose regex. The error type is `ArcLLMConfigError` and the message
starts with `"Invalid classification format"`. The next cell is a
**`# this raises`** demonstration.

In [ ]:
async def demo_format_rejection() -> None:
    msgs = [Message(role="user", content="hi")]
    # Fullwidth Latin "premium" — visually similar to "premium" but
    # contains characters outside the ASCII range. Reject before lookup.
    fullwidth = "\uff50\uff52\uff45\uff4d\uff49\uff55\uff4d"
    try:
        await router.invoke(msgs, classification=fullwidth)
    except ArcLLMConfigError as exc:
        print(type(exc).__name__, "→", exc)


await demo_format_rejection()

Same outcome for an uppercase variant — `"CUI"` is byte-different from `"cui"`, so even though there's a route named `cui` it never gets a chance to match.

In [ ]:
async def demo_case_rejection() -> None:
    msgs = [Message(role="user", content="hi")]
    try:
        await router.invoke(msgs, classification="CUI")
    except ArcLLMConfigError as exc:
        print(type(exc).__name__, "→", exc)


await demo_case_rejection()

And the zero-width space case — looks identical in a terminal but is a different byte sequence.

In [ ]:
async def demo_zwsp_rejection() -> None:
    msgs = [Message(role="user", content="hi")]
    payload = "cui\u200b"  # "cui" followed by U+200B ZERO WIDTH SPACE
    try:
        await router.invoke(msgs, classification=payload)
    except ArcLLMConfigError as exc:
        print(type(exc).__name__, "→", exc)
        print(f"(payload byte length: {len(payload.encode('utf-8'))})")


await demo_zwsp_rejection()

## 6. Enforcement modes — `"block"` vs `"warn"`

Once the format check passes, the router looks up the adapter. If
the classification is well-formed but **unknown** (no matching key
in `adapters`), `enforcement` decides what happens:

| Mode | Unknown classification → | Use when |
|---|---|---|
| `"block"` (default) | raises `ArcLLMConfigError` | Federal / regulated. Better to fail loudly than route sensitive data to a wrong provider. |
| `"warn"` | logs a `WARNING` and routes to `default_classification` | Development, tier-mixed personal use. The default route is known to be safe; missing tags get the safe path. |

There is **no permissive / silent / strict** alias — the source has
exactly two modes. Anything else at construction time is rejected:

```python
if self._enforcement not in ("warn", "block"):
    raise ArcLLMConfigError(
        f"enforcement must be 'warn' or 'block', got '{self._enforcement}'"
    )
```

Constructor rejects anything other than `"warn"` or `"block"`. **`# this raises`** below.

In [ ]:
try:
    RoutingModule(
        {"enforcement": "strict", "default_classification": "unclassified"},
        {"unclassified": make_adapter("openai", "gpt-4o-mini")},
    )
except ArcLLMConfigError as exc:
    print(type(exc).__name__, "→", exc)

`"block"` mode — unknown classification raises and **no adapter gets called**. This is the federal-default behaviour.

In [ ]:
async def demo_block_mode() -> None:
    cui = make_adapter("anthropic", "claude")
    unc = make_adapter("openai", "gpt")
    block_router = RoutingModule(
        {"enforcement": "block", "default_classification": "unclassified"},
        {"cui": cui, "unclassified": unc},
    )
    msgs = [Message(role="user", content="sensitive payload")]
    try:
        await block_router.invoke(msgs, classification="secret")
    except ArcLLMConfigError as exc:
        print(type(exc).__name__, "→", exc)
    print("\ncui adapter calls:", cui.invoke.await_count)
    print("unc adapter calls:", unc.invoke.await_count)
    print("Neither adapter was called — payload never left the router.")


await demo_block_mode()

`"warn"` mode — unknown classification falls back to the default route. The router emits a `WARNING` log so this never happens silently. Capture the log so we can see it.

In [ ]:
import io
import logging

# Capture log records emitted by the routing module
buf = io.StringIO()
handler = logging.StreamHandler(buf)
handler.setLevel(logging.WARNING)
routing_logger = logging.getLogger("arcllm.modules.routing")
routing_logger.addHandler(handler)
routing_logger.setLevel(logging.WARNING)


async def demo_warn_mode() -> None:
    cui = make_adapter("anthropic", "claude")
    unc = make_adapter("openai", "gpt")
    warn_router = RoutingModule(
        {"enforcement": "warn", "default_classification": "unclassified"},
        {"cui": cui, "unclassified": unc},
    )
    msgs = [Message(role="user", content="hi")]
    result = await warn_router.invoke(msgs, classification="secret.squirrel")
    print("response:", result.content)
    print("cui adapter calls:", cui.invoke.await_count)
    print("unc adapter calls:", unc.invoke.await_count, "  ← defaulted here")


await demo_warn_mode()

routing_logger.removeHandler(handler)
print("\n--- captured WARNING log ---")
print(buf.getvalue() or "(no warning captured — see above)")

Note the OpenTelemetry side-effects too. Even though we haven't
configured an exporter in this notebook, the router calls
`set_attribute` on a span for each invocation:

- `arcllm.routing.classification` — the requested tag
- `arcllm.routing.enforcement` — `"block"` or `"warn"`
- `arcllm.routing.action` — `"routed"` if hit, `"defaulted"` if missed in warn mode
- `arcllm.routing.selected_provider` / `selected_model` — what got picked

So every routing decision is observable downstream when an OTel SDK
is configured (see notebook 11).

## 7. Multi-provider routing — three real-world routes

Build a richer router with three classifications, each pointing at a
different mock provider:

- `premium` → mock Anthropic (best quality, expensive)
- `fast` → mock Groq (cheap, low-latency)
- `cui` → mock Azure OpenAI Government (compliant, the default)

Each is keyed on a different classification name, so we can
demonstrate dispatch by tag and verify the correct adapter is chosen.

In [ ]:
premium = make_adapter("anthropic", "claude-opus-4-1")
fast = make_adapter("groq", "llama-3.3-70b")
cui = make_adapter("azure_openai", "gpt-4o-fedramp")

multi_router = RoutingModule(
    {"enforcement": "block", "default_classification": "cui"},
    {"premium": premium, "fast": fast, "cui": cui},
)

print("Configured routes:")
for tag, adapter in multi_router._adapters.items():
    print(f"  {tag:12} → {adapter.name:18} {adapter.model_name}")
print("\nDefault route:", multi_router.name, "/", multi_router.model_name)

Run a small batch of requests with different classifications and watch each one land on the right adapter.

In [ ]:
async def run_batch() -> None:
    msgs = [Message(role="user", content="batch")]
    for tag in ("premium", "fast", "cui", "premium", "fast"):
        await multi_router.invoke(msgs, classification=tag)


await run_batch()

print("Dispatch counts after 5 calls (premium x2, fast x2, cui x1):")
print(f"  premium adapter: {premium.invoke.await_count}")
print(f"  fast adapter:    {fast.invoke.await_count}")
print(f"  cui adapter:     {cui.invoke.await_count}")

Tools and other kwargs flow through to whichever adapter wins the dispatch. Confirm by sending a `Tool` with `classification=fast` and inspecting the captured call args on the `fast` adapter.

In [ ]:
calc_tool = Tool(
    name="calculator",
    description="Add two numbers",
    parameters={"a": "number", "b": "number"},
)


async def demo_tool_passthrough() -> None:
    msgs = [Message(role="user", content="2 + 2?")]
    fast.invoke.reset_mock()
    await multi_router.invoke(msgs, [calc_tool], classification="fast", max_tokens=64)

    args, kwargs = fast.invoke.call_args
    # positional: messages, tools
    msgs_arg, tools_arg = args
    print("fast adapter received:")
    print("  messages:", [m.role for m in msgs_arg])
    print("  tools:   ", [t.name for t in tools_arg] if tools_arg else None)
    print("  kwargs:  ", kwargs)


await demo_tool_passthrough()

Defensive copy of the adapters dict — mutate the input *after* construction and confirm the router is unaffected. This matters because rules in real deployments come from TOML loaders that may keep references to the same dict; the router must be immune to post-init drift.

In [ ]:
adapters_in: dict[str, MagicMock] = {
    "cui": make_adapter("azure", "gpt-4o-fed"),
    "unclassified": make_adapter("openai", "gpt-4o-mini"),
}
copy_router = RoutingModule(
    {"enforcement": "block", "default_classification": "unclassified"},
    adapters_in,
)

# After init: try to inject and try to remove
adapters_in["evil"] = make_adapter("evil", "model")
del adapters_in["cui"]

print("input dict now has  :", sorted(adapters_in.keys()))
print("router still has    :", sorted(copy_router._adapters.keys()))
print("router is immune to post-init mutation.")

## 8. Failure modes — what breaks where

Every failure path the router surfaces, and which exception you'll
see for each:

| Failure | Where caught | Exception | Adapter called? |
|---|---|---|---|
| Invalid `enforcement` value | `__init__` | `ArcLLMConfigError` | n/a — never built |
| Empty `adapters` dict | `__init__` | `ArcLLMConfigError` | n/a |
| `default_classification` not in `adapters` | `__init__` | `ArcLLMConfigError` | n/a |
| Malformed classification kwarg | `invoke()` (regex check) | `ArcLLMConfigError` | no |
| Unknown classification, `block` mode | `invoke()` (lookup) | `ArcLLMConfigError` | no |
| Unknown classification, `warn` mode | `invoke()` | (none — fall through to default) | yes — default |
| Downstream adapter raises | propagated | whatever the adapter raised | yes |
| `close()` — one adapter raises | `close()` | `ExceptionGroup` (built-in) | n/a — closing |

Walk each one.

Empty adapters → `__init__` rejects.

In [ ]:
try:
    RoutingModule(
        {"enforcement": "block", "default_classification": "unclassified"},
        {},
    )
except ArcLLMConfigError as exc:
    print(type(exc).__name__, "→", exc)

`default_classification` doesn't reference an existing adapter → `__init__` rejects.

In [ ]:
try:
    RoutingModule(
        {"enforcement": "block", "default_classification": "phantom"},
        {"cui": make_adapter("anthropic", "claude")},
    )
except ArcLLMConfigError as exc:
    print(type(exc).__name__, "→", exc)

Downstream adapter raises mid-`invoke()`. The router does not catch
or rewrite — exceptions propagate to whichever module wraps the
router (typically `RetryModule`, which can react).

In [ ]:
class _BoomError(RuntimeError):
    pass


broken = make_adapter("groq", "llama")
broken.invoke = AsyncMock(side_effect=_BoomError("503 from upstream"))

bad_router = RoutingModule(
    {"enforcement": "block", "default_classification": "unclassified"},
    {"unclassified": broken},
)


async def demo_propagation() -> None:
    try:
        await bad_router.invoke([Message(role="user", content="hi")])
    except _BoomError as exc:
        print(f"propagated: {type(exc).__name__} → {exc}")


await demo_propagation()

`close()` semantics. The router closes **every** adapter, even if
an earlier one raises. Failures are collected and re-raised as a
single `ExceptionGroup` (Python 3.11+). This guarantees no resource
leaks even if one provider's connection pool is misbehaving.

In [ ]:
good_a = make_adapter("anthropic", "claude")
good_b = make_adapter("openai", "gpt-4o-mini")
flaky = make_adapter("groq", "llama")
flaky.close = AsyncMock(side_effect=RuntimeError("conn reset"))

close_router = RoutingModule(
    {"enforcement": "block", "default_classification": "unclassified"},
    {"premium": good_a, "unclassified": good_b, "fast": flaky},
)


async def demo_close_resilience() -> None:
    try:
        await close_router.close()
    except ExceptionGroup as eg:
        print(f"raised ExceptionGroup with {len(eg.exceptions)} sub-exception(s)")
        for sub in eg.exceptions:
            print(f"  - {type(sub).__name__}: {sub}")
    print("\nclose call counts (should all be 1):")
    for tag, adapter in close_router._adapters.items():
        print(f"  {tag:14}: {adapter.close.await_count}")


await demo_close_resilience()

`validate_config()` aggregates — true only if every adapter validates true. One bad apple fails the whole router.

In [ ]:
ok = make_adapter("openai", "gpt-4o-mini")
broken_validate = make_adapter("anthropic", "claude")
broken_validate.validate_config.return_value = False

mixed = RoutingModule(
    {"enforcement": "block", "default_classification": "unclassified"},
    {"unclassified": ok, "cui": broken_validate},
)

print(
    "router.validate_config() =",
    mixed.validate_config(),
    " ← False because cui adapter reports invalid config",
)

## 9. Composition — where routing sits in the load_model stack

From `registry.load_model()`, the documented order is (outermost first):

```
Otel → Queue → Telemetry → CircuitBreaker → Audit → Security →
  Retry → Fallback → RateLimit → [Router | Adapter]
```

The router replaces the bare adapter at the **innermost** position.
Two consequences worth pinning down:

1. **The `classification` kwarg flows through every wrapping module
   first**, then gets popped at the router. Telemetry, retry,
   circuit breaker — none of them know about classification; they
   don't need to. They forward `**kwargs` blindly. That's why
   adding a router never broke any existing module.
2. **Per-call cost tracking still works** because telemetry wraps
   the router. The token usage and price come back up the stack
   from whichever adapter was actually selected, and the
   `cost_usd` field on `LLMResponse` reflects that adapter's
   pricing.

Demonstrate by building a tiny stack: `TelemetryModule(RoutingModule(...))`.

In [ ]:
from arcllm.modules.telemetry import TelemetryModule, clear_budgets

clear_budgets()  # start fresh

cui = make_adapter("anthropic", "claude-sonnet-4-6")
unc = make_adapter("openai", "gpt-4o-mini")

inner_router = RoutingModule(
    {"enforcement": "block", "default_classification": "unclassified"},
    {"cui": cui, "unclassified": unc},
)
stack = TelemetryModule(
    {"cost_input_per_1m": 3.00, "cost_output_per_1m": 15.00},
    inner_router,
)

print("stack         =", type(stack).__name__)
print("stack._inner  =", type(stack._inner).__name__)
print("\nTop-level identity comes from the router's *default* route:")
print("  stack.name        =", stack.name)
print("  stack.model_name  =", stack.model_name)

Send two calls — one CUI, one default — through the stack and verify telemetry computed cost while routing chose the right adapter.

In [ ]:
async def demo_stack() -> None:
    msgs = [Message(role="user", content="hello")]
    r1 = await stack.invoke(msgs, classification="cui")
    r2 = await stack.invoke(msgs)  # no classification → default

    print("CUI call    : cost_usd =", r1.cost_usd, " content =", r1.content)
    print("Default call: cost_usd =", r2.cost_usd, " content =", r2.content)

    print("\nadapter call counts after stack run:")
    print(f"  cui adapter: {cui.invoke.await_count}")
    print(f"  unc adapter: {unc.invoke.await_count}")

    # Verify classification was popped — adapter never sees it
    assert "classification" not in cui.invoke.call_args.kwargs
    assert "classification" not in unc.invoke.call_args.kwargs
    print("\nclassification kwarg correctly stripped before reaching adapters.")


await demo_stack()

`close()` cascades through the stack: closing telemetry calls into the router, which closes every routed adapter.

In [ ]:
async def demo_stack_close() -> None:
    cui.close.reset_mock()
    unc.close.reset_mock()
    await stack.close()
    print("After stack.close():")
    print(f"  cui.close awaited:  {cui.close.await_count}")
    print(f"  unc.close awaited:  {unc.close.await_count}")


await demo_stack_close()

From a `load_model()` perspective, the same composition is achieved
with TOML rather than hand-built adapters. Excerpt from
`arcllm/config.toml`:

```toml
[modules.routing]
enabled = true
enforcement = "block"
default_classification = "unclassified"

[modules.routing.rules.cui]
provider = "anthropic"
model = "claude-sonnet-4-6"

[modules.routing.rules.unclassified]
provider = "openai"
model = "gpt-4o-mini"
```

And calling code:

```python
model = load_model("anthropic", routing=True)
resp = await model.invoke(messages, classification="cui")
```

Notebook 06 walks through that wiring. This notebook deliberately
stays close to the `RoutingModule` constructor so you can see what
the registry is doing for you.

## 10. Summary

**Recap:**

- `RoutingModule` is a switchboard at the innermost stack position. It
  takes a dict of pre-built adapters keyed by classification name, and
  dispatches each `invoke()` to whichever adapter matches the
  request's `classification` kwarg.
- Two enforcement modes — `"block"` (default; raise on miss) and
  `"warn"` (log + fall back to `default_classification`). No third option.
- Classifications are validated against a strict ASCII regex
  `^[a-z][a-z0-9_:.\-]{0,127}$`. Lowercase only, no whitespace, no
  Unicode. This blocks homoglyph attacks (fullwidth Latin, Cyrillic
  lookalikes, zero-width injection) before any adapter lookup happens.
- The `classification` kwarg is *popped* by the router and never
  reaches the chosen adapter. Other kwargs (tools, max_tokens, etc.)
  pass through untouched.
- The router maintains a defensive copy of the adapters dict, so
  post-init mutations of the input dict can't inject or remove routes.
- `validate_config()` is a hard AND across all adapters; one bad
  apple fails the whole router.
- `close()` closes every adapter even if individual ones raise — and
  collects failures into an `ExceptionGroup` for the caller.
- Routing composes cleanly under telemetry, retry, circuit breaker
  and the rest of the stack because every wrapping module forwards
  `**kwargs` blindly.

**API surface covered:**

- `arcllm.modules.routing.RoutingModule` — class, `__init__(config, adapters)`
- `RoutingModule.invoke(messages, tools, **kwargs)` — `classification` kwarg consumption
- `RoutingModule.name` / `RoutingModule.model_name` — surface the default route
- `RoutingModule.validate_config()` — aggregate validation
- `RoutingModule.close()` — fan-out close with `ExceptionGroup` aggregation
- The internal `_CLASSIFICATION_RE` pattern (private, but documented above)
- `arcllm.exceptions.ArcLLMConfigError` — raised on every config violation

**Next:**

- Notebook **06** — how `load_model(routing=...)` wires this from
  TOML and per-call config.
- Notebook **07** — the full module stack and why the router lives
  at the innermost position.
- Notebook **04** — every other `load_model` kwarg (`retry`,
  `fallback`, `circuit_breaker`, `queue`, ...) that wraps the router.